## Loading the Dataset

In [1]:
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD


In [2]:

recipe2M = pd.read_csv('recipes_data.csv')

In [3]:
recipe2M.head()

,title,ingredients,directions,link,source,NER,site
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""bite size shredded rice biscuits"", ""vanilla""...",www.cookbooks.com
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""cream of mushroom soup"", ""beef"", ""sour cream...",www.cookbooks.com
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...",www.cookbooks.com
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken gravy"", ""cream of mushroom soup"", ""c...",www.cookbooks.com
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""graham cracker crumbs"", ""powdered sugar"", ""p...",www.cookbooks.com


## Removing unnecessary columns & nulls

In [4]:
recipe2M_cleaned=recipe2M.drop(columns=['link', 'source', 'site'], inplace=False)
recipe2M_cleaned.dropna()

,title,ingredients,directions,NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p..."
...,...,...,...,...
2231137,Sunny's Fake Crepes,"[""1/2 cup chocolate hazelnut spread (recommend...","[""Spread hazelnut spread on 1 side of each tor...","[""chocolate hazelnut spread"", ""marshmallows"", ..."
2231138,Devil Eggs,"[""1 dozen eggs"", ""1 paprika"", ""1 salt and pepp...","[""Boil eggs on medium for 30mins."", ""Then cool...","[""choice"", ""miracle whip"", ""eggs"", ""relish"", ""..."
2231139,Extremely Easy and Quick - Namul Daikon Salad,"[""150 grams Daikon radish"", ""1 tbsp Sesame oil...","[""Julienne the daikon and squeeze out the exce...","[""soy sauce"", ""radish"", ""white sesame seeds"", ..."
2231140,Pan-Roasted Pork Chops With Apple Fritters,"[""1 cup apple cider"", ""6 tablespoons sugar"", ""...","[""In a large bowl, mix the apple cider with 4 ...","[""apple cider"", ""egg"", ""sugar"", ""freshly groun..."


## Removing recipes with directions contatining the word "step"

In [5]:

recipe2M_cleaned = recipe2M_cleaned[~recipe2M_cleaned['directions'].str.contains('step', case=False, na=False)]
recipe2M_cleaned['title'].count()


2206617

## Removing recipes with at most 1 ingredient

In [6]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['ingredients'].apply(lambda x: len([i for i in x if i.strip()]) <= 1)].index, inplace=True)
recipe2M_cleaned['title'].count()


2206617

## Removing recipes with instructions less than 10 characters

In [7]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['directions'].apply(lambda x: not all(len(i.strip()) < 10 for i in x if i.strip()))].index, inplace=True)
recipe2M_cleaned['title'].count()

2206617

## Removing recipes with title less than 4 characters 

In [8]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['title'].apply(lambda x: len(str(x)) < 4 if pd.notnull(x) else False)].index, inplace=True)
recipe2M_cleaned['title'].count()

2206372

## Extract raw ingredients

In [9]:
recipes = recipe2M_cleaned

In [10]:
# Tokenize a string into words
recipes['tokens'] = recipes['NER'].apply(word_tokenize)


In [11]:
#adding customized stop words
irrelevant_words = {
    'fresh', 'frozen', 'thawed', 'raw', 'grated', 'diced', 'chopped', 'minced',
    'powdered', 'sliced', 'ground', 'cooked', 'boiled', 'roasted', 'steamed',
    'baked', 'fried', 'toasted', 'crushed', 'peeled', 'skinned', 'shredded',
    'melted', 'whipped', 'pinch', 'dash', 'handful', 'cup', 'tablespoon',
    'teaspoon', 'liter', 'ml', 'oz', 'lb', 'gram', 'kg', 'quart', 'optional',
    'to taste', 'as needed', 'prepared', 'ready-made', 'store-bought', 'homemade',
    'pre-cooked', 'large', 'small', 'medium', 'whole', 'half', 'quartered',
    'extra', 'light', 'dark', 'white', 'black', 'red', 'green', 'yellow',
    'brown', 'golden', 'sweet', 'bitter', 'spicy', 'mild', 'hot', 'cold',
    'water', 'broth', 'stock', 'sauce', 'seasoning', 'marinade','bite','size'
}
stop_words = set(stopwords.words('english'))
stop_words.update(irrelevant_words)

In [12]:
lemmatizer = WordNetLemmatizer()

# Function to lemmatize nouns
def lemmatize(word, pos):
    if pos.startswith('NN'):  
        return lemmatizer.lemmatize(word, pos='n')
    else:
        return word  


In [13]:
#apply stop words removal and lemmatization
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: [lemmatize(word.lower(), tag) for word, tag in nltk.pos_tag(x) if word.isalnum() and word.lower() not in stop_words]
)

In [14]:
#filter uninque ingredients
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: list(set(x))
)

In [15]:
print(recipes['tokens'])

0          [biscuit, nut, vanilla, rice, sugar, milk, but...
1          [breast, soup, cream, mushroom, beef, chicken,...
2          [powder, salt, cream, pepper, cheese, garlic, ...
3            [soup, cream, gravy, cheese, mushroom, chicken]
4          [peanut, graham, chip, cracker, crumb, sugar, ...
                                 ...                        
2231136    [powder, coconut, salt, carrot, curry, vegetab...
2231137    [butter, tortilla, marshmallow, hazelnut, spre...
2231139               [sesame, salt, seed, oil, soy, radish]
2231140    [unsalted, salt, berry, arbol, neutral, cider,...
2231141    [white, salt, pepper, sausage, cheese, paste, ...
Name: tokens, Length: 2206373, dtype: object


In [16]:
#adding ids to recipes
recipes['recipe_id'] = recipes.index + 1

In [17]:
#reorder the columns
columns = ['recipe_id'] + [col for col in recipes.columns if col != 'recipe_id']
recipes = recipes[columns]

In [18]:
recipes.rename(columns={'tokens': 'raw_ingredients'}, inplace=True)

In [19]:
recipes.head()

,recipe_id,title,ingredients,directions,NER,raw_ingredients
0,1,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""...","[biscuit, nut, vanilla, rice, sugar, milk, but..."
1,2,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream...","[breast, soup, cream, mushroom, beef, chicken,..."
2,3,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[powder, salt, cream, pepper, cheese, garlic, ..."
3,4,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c...","[soup, cream, gravy, cheese, mushroom, chicken]"
4,5,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p...","[peanut, graham, chip, cracker, crumb, sugar, ..."


# Extract Cooking Methods

In [20]:
recipes.head()

,recipe_id,title,ingredients,directions,NER,raw_ingredients
0,1,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""...","[biscuit, nut, vanilla, rice, sugar, milk, but..."
1,2,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream...","[breast, soup, cream, mushroom, beef, chicken,..."
2,3,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[powder, salt, cream, pepper, cheese, garlic, ..."
3,4,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c...","[soup, cream, gravy, cheese, mushroom, chicken]"
4,5,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p...","[peanut, graham, chip, cracker, crumb, sugar, ..."


In [21]:
recipes['directions']

0          ["In a heavy 2-quart saucepan, mix brown sugar...
1          ["Place chipped beef on bottom of baking dish....
2          ["In a slow cooker, combine all ingredients. C...
3          ["Boil and debone chicken.", "Put bite size pi...
4          ["Combine first four ingredients and press in ...
                                 ...                        
2231136    ["Cook the onion in butter in a medium saucepa...
2231137    ["Spread hazelnut spread on 1 side of each tor...
2231139    ["Julienne the daikon and squeeze out the exce...
2231140    ["In a large bowl, mix the apple cider with 4 ...
2231141    ["Preheat the oven to 350.", "In a bowl, mix t...
Name: directions, Length: 2206373, dtype: object

In [22]:
cooking_methods_glossary = [
    "bake", "steam", "fry", "grill", "roast", "boil", "sauté", "poach", "broil", "braise",
    "stew", "smoke", "microwave", "blanch", "deep-fry", "barbecue", "sear", "pressure-cook",
    "simmer", "stir-fry","baste","batter","beat","blend","carmelize","chop","cream","cube",
    "cure","dice","dissolve","drain","fold","granish","grate","grease","julienne","knead",
    "marinate","mash","mince","parboil","pare","peel","pinch","pit","plump","preheat","puree",
    "reduce","saute","scald","sear","shred","sift","skim","slice","thaw","toss","whip"
]

In [23]:
stop_words = set(stopwords.words('english'))

In [24]:
cooking_methods = []

for directions in recipes['directions']:
    if pd.isna(directions):
        cooking_methods.append(None)
    else:
        # Tokenize words
        words = word_tokenize(directions.lower())
        # Remove stopwords and non-alphabetic tokens
        filtered_words = [word for word in words if word not in stop_words and word.isalpha()]

        methods = set(filtered_words).intersection(cooking_methods_glossary)

        cooking_methods.append(list(methods))

recipes['cooking_methods'] = cooking_methods

In [25]:
recipes['cooking_methods'].head(20)

0                                      [boil]
1                               [cream, bake]
2                                          []
3                         [boil, cream, bake]
4                                          []
5     [grease, cream, drain, boil, microwave]
6                               [beat, cream]
7                                      [bake]
8                                    [simmer]
9                         [chop, whip, drain]
10                    [fold, dissolve, drain]
11                                         []
12                 [microwave, fry, barbecue]
13                             [cream, drain]
14                                         []
15                              [cream, boil]
16                                     [bake]
17                                     [toss]
18                                    [cream]
19                               [bake, sift]
Name: cooking_methods, dtype: object

## Clean rating dataset

In [10]:
train_rating= pd.read_csv('core-data-train_rating.csv')
test_rating = pd.read_csv('core-data-test_rating.csv')

NameError: name 'pd' is not defined

In [ ]:
#mapping train data with correct recipe id
unique_old_ids = sorted(train_rating['recipe_id'].unique())
new_ids = recipes['recipe_id'].tolist() 

mapping = {old: new for old, new in zip(unique_old_ids, new_ids)}
train_rating['recipe_id'] = train_rating['recipe_id'].map(mapping)

In [ ]:
#mapping test data with correct recipe id
unique_old_ids = sorted(test_rating['recipe_id'].unique())
new_ids = recipes['recipe_id'].tolist()  

mapping = {old: new for old, new in zip(unique_old_ids, new_ids)}
test_rating['recipe_id'] = test_rating['recipe_id'].map(mapping)

In [ ]:
train_rating.drop(columns=['dateLastModified'], inplace=True)
test_rating.drop(columns=['dateLastModified'], inplace=True)

In [32]:
recipes.to_csv('cleanedrecipes.csv', index=False)

In [34]:
train_rating.to_csv('cleanedTrainRating.csv',index=False)
test_rating.to_csv('cleanedTestRating.csv',index=False)

# Models

In [2]:
cleaned_recipes = pd.read_csv('/kaggle/input/late-plate/cleanedrecipes.csv')
cleaned_train_rating = pd.read_csv('/kaggle/input/rating/cleanedTrainRating.csv')
cleaned_test_rating = pd.read_csv('/kaggle/input/rating/cleanedTestRating.csv')

In [23]:
cleaned_train_rating

,user_id,recipe_id,rating
0,5215572,9088,5
1,5215572,25264,4
2,5215572,9138,5
3,3622615,17758,4
4,1313770,16467,5
...,...,...,...
676941,3023108,17491,5
676942,3023108,652,5
676943,3023108,26656,1
676944,3023108,1107,3


## Collaborative Filtering

In [4]:
# Ensure every user-recipe pair is unique
cleaned_train_rating.duplicated(subset=['user_id' , 'recipe_id']).sum()

0

In [11]:
import tensorflow as tf 
physical_devices = tf.config.list_physical_devices('GPU')
for device in physical_devices:
    tf.config.experimental.set_memory_growth(device, True)
print("Using GPU:", tf.config.list_physical_devices('GPU'))

# mapping ids
user_ids = cleaned_train_rating['user_id'].unique()
user_to_index = {user: idx for idx, user in enumerate(user_ids)}
num_users = len(user_ids)

recipe_ids = cleaned_train_rating['recipe_id'].unique()
recipe_to_index = {recipe: idx for idx, recipe in enumerate(recipe_ids)}
num_recipes = len(recipe_ids)

# Convert data to TensorFlow tensors
train_users = tf.constant(cleaned_train_rating['user_id'].map(user_to_index).values, dtype=tf.int64)
train_recipes = tf.constant(cleaned_train_rating['recipe_id'].map(recipe_to_index).values, dtype=tf.int64)
train_ratings = tf.constant(cleaned_train_rating['rating'].values, dtype=tf.float32)

Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [97]:
# Hyperparameters
k = 15             
learning_rate = 0.001
lambda_ = 0.02     # L2
epochs = 30
mu = cleaned_train_rating['rating'].mean().astype(np.float32)

# Initialize parameters on GPU
tf.random.set_seed(42)
P = tf.Variable(
    tf.random.normal([num_users, k], stddev=1/tf.sqrt(tf.cast(k, tf.float32))),
    trainable=True, name="User_Factors"
)
Q = tf.Variable(
    tf.random.normal([num_recipes, k], stddev=1/tf.sqrt(tf.cast(k, tf.float32))),
    trainable=True, name="Item_Factors"
)
b_u = tf.Variable(tf.zeros([num_users]), trainable=True, name="User_Biases")
b_i = tf.Variable(tf.zeros([num_recipes]), trainable=True, name="Item_Biases")

In [98]:
optimizer = tf.optimizers.Adam(learning_rate)

batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices((train_users, train_recipes, train_ratings))
dataset = dataset.shuffle(buffer_size=1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)

@tf.function
def train_step(u, i, r_ui):
    with tf.GradientTape() as tape:
        user_bias = tf.gather(b_u, u)
        item_bias = tf.gather(b_i, i)
        user_factor = tf.gather(P, u)
        item_factor = tf.gather(Q, i)
        
        pred = mu + user_bias + item_bias + tf.reduce_sum(user_factor * item_factor, axis=1)
        error = r_ui - pred
        loss = tf.reduce_mean(tf.square(error)) + lambda_ * (
            tf.reduce_sum(tf.square(user_factor)) + 
            tf.reduce_sum(tf.square(item_factor)) +
            tf.reduce_sum(tf.square(user_bias)) +
            tf.reduce_sum(tf.square(item_bias))
        )
        
    grads = tape.gradient(loss, [P, Q, b_u, b_i])
    optimizer.apply_gradients(zip(grads, [P, Q, b_u, b_i]))
    return loss

# Training loop
for epoch in range(epochs):
    epoch_loss = 0.0
    num_batches = 0
    for u, i, r_ui in dataset:
        loss = train_step(u, i, r_ui)
        epoch_loss += loss
        num_batches += 1
    print(f"Epoch {epoch+1}: Loss = {epoch_loss/num_batches:.4f}")

Epoch 1: Loss = 3.4589
Epoch 2: Loss = 1.5275
Epoch 3: Loss = 1.0443
Epoch 4: Loss = 0.8750
Epoch 5: Loss = 0.7996
Epoch 6: Loss = 0.7599
Epoch 7: Loss = 0.7365
Epoch 8: Loss = 0.7218
Epoch 9: Loss = 0.7120
Epoch 10: Loss = 0.7052
Epoch 11: Loss = 0.7004
Epoch 12: Loss = 0.6970
Epoch 13: Loss = 0.6944
Epoch 14: Loss = 0.6926
Epoch 15: Loss = 0.6912
Epoch 16: Loss = 0.6902
Epoch 17: Loss = 0.6894
Epoch 18: Loss = 0.6888
Epoch 19: Loss = 0.6884
Epoch 20: Loss = 0.6881
Epoch 21: Loss = 0.6878
Epoch 22: Loss = 0.6876
Epoch 23: Loss = 0.6874
Epoch 24: Loss = 0.6873
Epoch 25: Loss = 0.6872
Epoch 26: Loss = 0.6872
Epoch 27: Loss = 0.6871
Epoch 28: Loss = 0.6870
Epoch 29: Loss = 0.6870
Epoch 30: Loss = 0.6870


In [100]:
# Evaluate on test set
test_users = cleaned_test_rating['user_id'].map(user_to_index).values
test_recipes = cleaned_test_rating['recipe_id'].map(recipe_to_index).values
test_ratings = cleaned_test_rating['rating'].values

# Handle missing users/items
valid_mask = np.isin(test_users, list(user_to_index.values())) & np.isin(test_recipes, list(recipe_to_index.values()))
test_users = test_users[valid_mask]
test_recipes = test_recipes[valid_mask]
test_ratings = test_ratings[valid_mask]

# Convert to tensors
test_users = tf.constant(test_users, dtype=tf.int64)
test_recipes = tf.constant(test_recipes, dtype=tf.int64)

# Compute predictions
user_bias_test = tf.gather(b_u, test_users)
item_bias_test = tf.gather(b_i, test_recipes)
user_factor_test = tf.gather(P, test_users)
item_factor_test = tf.gather(Q, test_recipes)

test_preds = mu + user_bias_test + item_bias_test + tf.reduce_sum(user_factor_test * item_factor_test, axis=1)
test_rmse = tf.sqrt(tf.reduce_mean((test_ratings - test_preds.numpy()) ** 2))

print(f"\nTest RMSE: {test_rmse.numpy():.4f}")


Test RMSE: 0.8552


In [101]:
def get_top_recommendations(user_id, top_n=5):
    # Check if the user exists in our mapping
    if user_id not in user_to_index:
        print("User not found in training data.")
        return []
    
    # Get the internal index for this user
    user_idx = user_to_index[user_id]
    
    # Compute predicted ratings for all recipes
    user_bias = b_u[user_idx]
    dot_product = tf.reduce_sum(P[user_idx] * Q, axis=1)
    pred = mu + user_bias + b_i + dot_product
    
    # Convert predictions to a NumPy array
    pred_np = pred.numpy()
    
    # Exclude already rated recipes
    user_rated = cleaned_train_rating[cleaned_train_rating['user_id'] == user_id]['recipe_id'].values
    # Map the rated recipe ids to internal indices.
    already_rated_indices = {recipe_to_index[recipe] for recipe in user_rated if recipe in recipe_to_index}
    
    # Set the predictions for already rated recipes to -infinity.
    for idx in already_rated_indices:
        pred_np[idx] = -np.inf
    
    # Get the indices of the top_n highest predicted ratings
    top_indices = pred_np.argsort()[-top_n:][::-1]
    
    # Create an inverse mapping for recipes: index -> original recipe id
    index_to_recipe = {idx: recipe for recipe, idx in recipe_to_index.items()}
    
    # Map the top indices back to the original recipe IDs
    top_recipes = [index_to_recipe[idx] for idx in top_indices]
    
    return top_recipes

In [102]:
recommended_recipes = get_top_recommendations(5215572)
print("Top 5 Recommendations:", recommended_recipes)

Top 5 Recommendations: [24014, 2434, 8546, 28653, 16352]


## Popularity Based

In [58]:
# Calculate average rating and count
popularity_avg = cleaned_train_rating.groupby('recipe_id')['rating'].agg(['mean', 'count']).reset_index()
popularity_avg.columns = ['recipe_id', 'avg_rating', 'num_ratings']

# Filter recipes with at least 3 ratings 
min_ratings = 3
popularity_avg_filtered = popularity_avg[popularity_avg['num_ratings'] >= min_ratings]

# Rank by highest average rating
top_rated = popularity_avg_filtered.sort_values(by='avg_rating', ascending=False)
print("\nHighest Average-Rated Recipes (with min 2 ratings):")
print(top_rated.head())


Highest Average-Rated Recipes (with min 2 ratings):
       recipe_id  avg_rating  num_ratings
28873      28899         5.0            4
28936      28962         5.0            3
28871      28897         5.0            3
29088      29114         5.0            4
28826      28852         5.0            3


In [59]:
# Weighted score 
min_ratings_for_weight = 1 
popularity_avg['weighted_score'] = (popularity_avg['avg_rating'] * popularity_avg['num_ratings']) / (popularity_avg['num_ratings'] + min_ratings_for_weight)

# Rank by weighted score
top_hybrid = popularity_avg.sort_values(by='weighted_score', ascending=False)
print("\nHybrid Popularity (Weighted Score):")
print(top_hybrid.head())


Hybrid Popularity (Weighted Score):
       recipe_id  avg_rating  num_ratings  weighted_score
27167      27192    4.952941           85        4.895349
19768      19790    4.906897          290        4.890034
5584        5592    4.901163          344        4.886957
19090      19111    4.904762          252        4.885375
4367        4373    4.913669          139        4.878571


In [61]:
def recommend_popular_recipes(popularity_df, n=10):
    return popularity_df['recipe_id'].tolist()[:n]

recommendations = recommend_popular_recipes(top_hybrid, n=5)
print("\nTop Recommendations:", recommendations)


Top Recommendations: [27192, 19790, 5592, 19111, 4373]
